# ResNet — почему skip-connection чинит глубину

**Статья:** He, Zhang, Ren, Sun — *Deep Residual Learning for Image Recognition* (CVPR 2016, arXiv:1512.03385)

🌳 **Узел дерева:** DL foundations → optimization → residual learning

**Что воспроизводим:** degradation problem и то, как его снимает identity-skip. Контраст один и чистый: `plain-deep` (`y = block(x)`) против `residual-deep` (`y = block(x) + x`). Всё остальное — глубина, ширина, инициализация, BN, оптимизатор, эпохи, seed — идентично. Эффект статьи (Fig. 4): глубокая plain-сеть тренируется ХУЖЕ мелкой, а skip это чинит.

**Данные:** sklearn `digits` (1797 сэмплов, 8×8, 10 классов). CIFAR ResNet-110 на CPU за минуты не обучить, но механизм degradation архитектурно-агностичен — берём глубокий MLP на реальных данных.

**Компьют:** CPU, несколько эпох, пара минут. Обучение настоящее.

**Метрика:** TRAINING accuracy как основная величина — degradation это про оптимизацию, не про генерализацию. Test acc печатаем санити-чеком.

**Parts:** (1) residual-блок — вклад статьи · (2) He/MSRA init · (3) training loop · (4) метрика train/test · (5) ablation-свип по глубине + фигура.

## ⛏ Эксперименты-первыми (заполни ДО чтения метода)

Статья большая, но сегодня один сюжет — Fig. 1 и Fig. 4 (CIFAR): plain-56 даёт БОЛЬШУЮ training error, чем plain-20.

> **(a) Что Fig. 4 реально показала vs то, что обещает abstract про «ultra-deep»?**
> …

> **(b) Чем бы фальсифицировала «это не переобучение»? Какую кривую надо посмотреть?**
> …

> **(c) Option A (identity) vs option B (projection): где авторы честно признают, что разница мелкая и почему это важно для «вклада»?**
> …

## Зачем это

Наивная интуиция: больше слоёв — не хуже, в пределе лишние слои выучат тождество. На практике — нет. Глубокая plain-сеть с BN и нормальной инициализацией всё равно оптимизируется хуже мелкой: training error РАСТЁТ с глубиной. Это не vanishing gradient (BN его снимает) и не переобучение (страдает именно train). Авторы называют это degradation.

**Фишка.** Вместо того чтобы просить стек слоёв выучить целевое отображение $H(x)$, пусть он учит остаток $F(x) = H(x) - x$, а мы восстанавливаем $H(x) = F(x) + x$ через identity-shortcut без единого лишнего параметра. Если оптимум близок к тождеству, загнать $F \to 0$ проще, чем собрать тождество из стека нелинейностей с нуля. Один плюс в forward — и очень глубокие сети снова оптимизируемы.

**Почему digits, а не CIFAR.** Механизм не про свёртки — про то, как остаточная параметризация меняет ландшафт оптимизации. Глубокий MLP на реальных цифрах ловит тот же эффект за минуты на CPU, а контраст plain/residual остаётся чистой абляцией одной идеи.

## Теория (Фейнман)

**Задача.** Стек блоков должен реализовать $H(x)$. В plain-параметризации блок сам и есть $H$: сеть настраивает веса так, чтобы композиция блоков дала нужное отображение. Эмпирически при большой глубине SGD не находит даже тождество — добавь слои, и train error лезет вверх.

**Механизм.** Перепараметризуем блок. Пусть внутренние слои считают $F(x)$, а выход блока — $$y = F(x) + x.$$ Тогда «ничего не делать» = $F(x) = 0$, что достигается занулением весов последнего слоя (после BN — сдвигом в ноль). Тождество стало точкой, до которой рукой подать, а не целью обучения.

**Что метод измеряет.** Разницу в *training* accuracy между plain-deep и residual-deep при идентичном всём остальном. Положительный разрыв = вклад воспроизвёлся. Формально смотрим $\text{gap} = \text{acc}^{train}_{res} - \text{acc}^{train}_{plain}$ и деградацию $\text{acc}^{train}_{plain} < \text{acc}^{train}_{shallow}$.

*Где у меня дыра в понимании:* …

❓ **Inline Question 1.** BN уже стабилизирует градиенты и статистики. Почему тогда plain-deep всё равно деградирует — что именно skip добавляет поверх BN, чего BN не даёт?

**Ваш ответ:** …

In [ ]:
# --- setup (готово, просто запусти) ---
# pip install numpy torch scikit-learn matplotlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cpu")
SEED = 0

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)

def load_data():
    d = load_digits()
    X, y = d.data.astype("float32"), d.target.astype("int64")
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
    scaler = StandardScaler().fit(Xtr)
    Xtr = scaler.transform(Xtr).astype("float32")
    Xte = scaler.transform(Xte).astype("float32")
    return (torch.from_numpy(Xtr), torch.from_numpy(ytr),
            torch.from_numpy(Xte), torch.from_numpy(yte))

def majority_baseline(ytr, yte):
    maj = torch.bincount(ytr).argmax().item()
    acc = (yte == maj).float().mean().item()
    return maj, acc

set_seed(SEED)
Xtr, ytr, Xte, yte = load_data()
IN_DIM, N_CLASSES = Xtr.shape[1], int(ytr.max().item() + 1)
WIDTH = 64
maj_cls, maj_acc = majority_baseline(ytr, yte)
print(f"train={Xtr.shape[0]} test={Xte.shape[0]} in_dim={IN_DIM} classes={N_CLASSES} width={WIDTH}")
print(f"[floor] majority-class (predict {maj_cls}) test acc = {maj_acc:.3f}")

## Part 1 — residual-блок (вклад статьи)

Блок держит четыре готовых подслоя: `fc1, bn1, fc2, bn2` (width→width каждый). Это 2-слойный basic block из статьи (у них conv→conv, у нас fc→fc), с BN после каждого линейного и перед активацией.

Твоя задача — собрать forward так, чтобы при `residual=True` блок реализовывал остаточную параметризацию из теории: внутренние слои считают $F(x)$, а наружу идёт восстановленное $H(x) = F(x) + x$. При `residual=False` — та же цепочка без shortcut (plain-вариант для абляции). Финальная нелинейность — общая для обоих.

In [ ]:
class Block(nn.Module):
    """Один trunk-блок глубокого MLP. Флаг `residual` включает identity-shortcut
    (вклад статьи, option A — parameter-free).

    Подслои (заданы в __init__, width->width):
      fc1, bn1, fc2, bn2  — линейный + BatchNorm1d каждый.

    Требуется реализовать forward(x):
      x: [B, width] float.
      Возвращает: [B, width] float — выход блока.
      Внутренние слои реализуют F(x); при self.residual выход блока восстанавливает
      отображение H(x)=F(x)+x через parameter-free identity-shortcut (иначе — plain,
      без shortcut). Нелинейность на выходе применяется в обоих случаях одинаково.
    Note: width постоянна по всему стволу, поэтому x и F(x) складываются напрямую, без проекции.
    """
    def __init__(self, width, residual):
        super().__init__()
        self.residual = residual
        self.fc1 = nn.Linear(width, width)
        self.bn1 = nn.BatchNorm1d(width)
        self.fc2 = nn.Linear(width, width)
        self.bn2 = nn.BatchNorm1d(width)

    def forward(self, x):
        # YOUR CODE
        raise NotImplementedError

print("Block defined:", Block(WIDTH, residual=True))

In [ ]:
# Тест: при residual=True на нулевых весах последнего линейного слоя блок должен
# отдавать ~identity (после ReLU), т.е. F(x)=0 -> y=relu(x). Плюс проверяем, что
# residual и plain РАЗЛИЧАЮТСЯ на общем входе и grad течёт.
set_seed(0)
b_res = Block(WIDTH, residual=True).eval()
b_pln = Block(WIDTH, residual=False).eval()
b_pln.load_state_dict(b_res.state_dict())  # одинаковые веса -> разница только в skip

x = torch.randn(8, WIDTH)
yr = b_res(x); yp = b_pln(x)
assert yr.shape == (8, WIDTH), "форма выхода должна быть [B, width]"
assert not torch.allclose(yr, yp), "residual и plain совпали — skip не применяется?"

# F(x)=0 -> residual-блок = relu(x): занулим веса/сдвиги ВТОРОГО линейного и bn2.
with torch.no_grad():
    b_res.fc2.weight.zero_(); b_res.fc2.bias.zero_()
    b_res.bn2.weight.zero_(); b_res.bn2.bias.zero_()
b_res.eval()
y_id = b_res(x)
exp = F.relu(x)
err = (y_id - exp).abs().max().item()
print(f"Expected \u2248 relu(x) при F=0 | max|err| = {err:.2e}")
assert err < 1e-5, "при F(x)=0 residual-блок обязан давать relu(x); проверь, что складываешь ИМЕННО x"

# grad-flow
xg = torch.randn(8, WIDTH, requires_grad=True)
Block(WIDTH, residual=True).train()(xg).sum().backward()
assert xg.grad is not None and torch.isfinite(xg.grad).all(), "градиент по входу не течёт/не конечен"
print("ok")

❓ **Inline Question 2.** Тест зануляет `fc2`/`bn2` и ждёт `relu(x)`. Почему для «блок ничего не делает» достаточно занулить именно ВТОРОЙ линейный слой (а не первый), и что это говорит о том, насколько «дёшево» residual-стеку выучить тождество?

**Ваш ответ:** …

## Part 2 — He/MSRA инициализация

Статья (Sec. 3.2) инициализирует веса по He et al. 2015. Это не косметика: с плохой инициализацией plain-сеть умрёт сама по себе, и мы не отличим «degradation» от «взорвавшейся инициализации». He-init выравнивает дисперсию активаций под ReLU и изолирует эффект именно к skip-connection.

Реализуй `init_he(model)`: пройди по всем линейным подмодулям и переинициализируй их веса схемой He/MSRA под ReLU (режим по fan_in), а bias обнули. BN не трогай (дефолтная инициализация BN и так корректна).

In [ ]:
def init_he(model):
    """Инициализировать все nn.Linear внутри model по схеме He/MSRA (2015) под ReLU.

    Args:
      model: nn.Module — обходятся все вложенные подмодули.
    Returns:
      model (тот же объект, in-place): у каждого nn.Linear веса переинициализированы
      He/MSRA-схемой, согласованной с ReLU и fan_in; bias, если есть, обнулён.
      Слои BatchNorm не изменяются.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# He/MSRA под ReLU, fan_in: std(W) ~ sqrt(2/fan_in). Для width=64 -> ~0.177.
set_seed(0)
lin = nn.Linear(64, 64); nn.init.ones_(lin.bias)
m = nn.Sequential(lin, nn.BatchNorm1d(64))
init_he(m)
std = lin.weight.std().item()
expected = (2.0 / 64) ** 0.5
print(f"Expected std \u2248 {expected:.3f} | у тебя {std:.3f}")
assert abs(std - expected) < 0.05, "std весов не похож на sqrt(2/fan_in) — та ли схема/режим?"
assert torch.allclose(lin.bias, torch.zeros_like(lin.bias)), "bias должен быть обнулён"
print("ok")

## Собираем сеть (готово)

`DeepMLP` даётся целиком — это обвязка: stem, стек одинаковых блоков, линейная голова. `residual` флипает ТОЛЬКО skip; глубина/ширина/init общие. После сборки зовём твой `init_he`.

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, in_dim, width, n_blocks, n_classes, residual):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(in_dim, width), nn.BatchNorm1d(width), nn.ReLU())
        self.blocks = nn.ModuleList([Block(width, residual) for _ in range(n_blocks)])
        self.head = nn.Linear(width, n_classes)
        init_he(self)

    def forward(self, x):
        x = self.stem(x)
        for blk in self.blocks:
            x = blk(x)
        return self.head(x)

set_seed(0)
_m = DeepMLP(IN_DIM, WIDTH, 2, N_CLASSES, residual=True)
print("DeepMLP ok, out shape:", _m(Xtr[:4]).shape)

## Part 3 — метрика (train/test accuracy)

Degradation — про training accuracy, так что метрику меряем сами (и она нужна training-циклу в Part 4). Реализуй `evaluate`: перевести модель в eval, посчитать logits на train и test, вернуть cross-entropy-loss на train и accuracy на train и test.

Это именно та величина, по которой воспроизводится эффект статьи (train acc plain vs residual), — не proxy.

In [ ]:
@torch.no_grad()
def evaluate(model, Xtr, ytr, Xte, yte):
    """Посчитать метрики обученной модели БЕЗ градиентов, в режиме eval.

    Args:
      model: nn.Module -> logits [N, n_classes].
      Xtr, ytr, Xte, yte: train/test тензоры.
    Returns:
      dict с ключами:
        'train_loss': float — cross-entropy на train.
        'train_acc':  float — доля верных argmax-предсказаний на train (в [0,1]).
        'test_acc':   float — то же на test.
    Note: model переводится в eval-режим (BN использует накопленную статистику).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# Санити на необученной сети: ключи, диапазоны, и что train_loss согласуется с F.cross_entropy.
set_seed(0)
_probe = DeepMLP(IN_DIM, WIDTH, n_blocks=1, n_classes=N_CLASSES, residual=False)
out = evaluate(_probe, Xtr, ytr, Xte, yte)
assert set(out) == {"train_loss", "train_acc", "test_acc"}, "не те ключи в dict"
for k in ("train_acc", "test_acc"):
    assert 0.0 <= out[k] <= 1.0, f"{k} вне [0,1]"
_probe.eval()
with torch.no_grad():
    ref_loss = F.cross_entropy(_probe(Xtr), ytr).item()
print(f"Expected train_loss \u2248 {ref_loss:.4f} | у тебя {out['train_loss']:.4f}")
assert abs(out["train_loss"] - ref_loss) < 1e-4, "train_loss не совпал с прямым cross_entropy — та ли величина/аргументы?"
print("ok")

## Part 4 — training loop

Метод обучаемый, так что цикл пишешь сам. Статья: SGD с momentum, weight decay, cross-entropy, минибатчи. Оптимизатор уже создан, данные и число эпох переданы. Соберём цикл, который перемешивает train каждую эпоху, идёт минибатчами и делает шаг SGD по cross-entropy; в конце цикл зовёт твою `evaluate` из Part 3.

Важно: seed фиксируется ДО цикла (в стартовом коде), чтобы порядок данных был идентичен у всех вариантов — иначе абляция нечестная.

In [ ]:
def train_model(model, Xtr, ytr, Xte, yte, epochs=30, lr=0.03, batch=64):
    """Обучить model на (Xtr, ytr) минибатчевым SGD и вернуть метрики через evaluate.

    Args:
      model: nn.Module — классификатор, logits формы [B, n_classes].
      Xtr, ytr: [N, d] float, [N] int64 — train.
      Xte, yte: test (только для финального evaluate, не для обучения).
      epochs, lr, batch: int/float — гиперпараметры цикла.
    Returns:
      dict от evaluate(model, Xtr, ytr, Xte, yte) — метрики ПОСЛЕ обучения.
      Обучение: каждую эпоху train перемешивается заново; проход минибатчами
      размера batch; шаг оптимизатора по cross-entropy-лоссу на logits модели.
    Note: перемешивание и порядок батчей — единственный источник стохастики; seed
      уже зафиксирован ниже, не трогай его внутри цикла.
    """
    set_seed(SEED)  # идентичный порядок данных для каждого варианта -> честная абляция
    model.to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    n = Xtr.shape[0]
    # YOUR CODE
    raise NotImplementedError
    return evaluate(model, Xtr, ytr, Xte, yte)

In [ ]:
# Смоук: маленькая сеть за пару эпох обязана уверенно оторваться от majority-floor.
set_seed(0)
tiny = DeepMLP(IN_DIM, WIDTH, n_blocks=1, n_classes=N_CLASSES, residual=False)
mt = train_model(tiny, Xtr, ytr, Xte, yte, epochs=10)
print(f"Expected: train_acc сильно > majority ({maj_acc:.3f}) | у тебя train_acc={mt['train_acc']:.3f} test_acc={mt['test_acc']:.3f}")
assert mt["train_acc"] > 0.7, "мелкая сеть за 10 эпох не учится — крутишь ли ты opt.zero_grad/backward/step в правильном порядке?"
print("ok")

## Part 5 — ablation: degradation и как его чинит skip

Всё готово к главному контрасту. Три сети на одних данных, идентичный протокол, различие ровно в одном флаге:

- **shallow** (1 блок, plain) — планка «должно быть легко»;
- **plain-deep** (12 блоков, `y = block(x)`);
- **residual-deep** (12 блоков, `y = block(x) + x`).

12 блоков = $2 \cdot 12 + 2 = 26$ линейных слоёв — достаточно глубоко, чтобы plain деградировал. Смотрим на TRAIN accuracy.

In [ ]:
DEEP_BLOCKS = 12

set_seed(SEED)
shallow = DeepMLP(IN_DIM, WIDTH, n_blocks=1, n_classes=N_CLASSES, residual=False)
m_shallow = train_model(shallow, Xtr, ytr, Xte, yte)

set_seed(SEED)
plain_deep = DeepMLP(IN_DIM, WIDTH, DEEP_BLOCKS, N_CLASSES, residual=False)
m_plain = train_model(plain_deep, Xtr, ytr, Xte, yte)

set_seed(SEED)
res_deep = DeepMLP(IN_DIM, WIDTH, DEEP_BLOCKS, N_CLASSES, residual=True)
m_res = train_model(res_deep, Xtr, ytr, Xte, yte)

train_acc_gap = m_res["train_acc"] - m_plain["train_acc"]
n_layers = 2 * DEEP_BLOCKS + 2
print(f"[shallow]      train_acc={m_shallow['train_acc']:.3f} test_acc={m_shallow['test_acc']:.3f}")
print(f"[plain-{n_layers}L]     train_acc={m_plain['train_acc']:.3f} test_acc={m_plain['test_acc']:.3f}")
print(f"[residual-{n_layers}L]  train_acc={m_res['train_acc']:.3f} test_acc={m_res['test_acc']:.3f}")
print(f"residual - plain  train_acc gap = {train_acc_gap:+.3f}")

## Проверки эффекта статьи (headline)

Четыре assert-а маппят прямо на утверждения He et al. Пороги мягкие с большим запасом — наблюдаемый эффект в разы больше, они ловят присутствие эффекта, а не подогнаны под число.

In [ ]:
# 1. любой реальный классификатор бьёт majority-floor с большим отрывом
assert m_shallow["test_acc"] > maj_acc + 0.3, "shallow едва бьёт majority?!"
print(f"[ok] shallow бьёт floor на {m_shallow['test_acc']-maj_acc:+.3f}")

# 2. DEGRADATION: plain-deep тренируется ХУЖЕ мелкой сети (не переобучение — страдает train)
degraded = m_plain["train_acc"] < m_shallow["train_acc"] - 0.02
print(f"[{'ok' if degraded else '??'}] plain-deep train_acc ({m_plain['train_acc']:.3f}) vs shallow ({m_shallow['train_acc']:.3f}) -> {'DEGRADED' if degraded else 'нет явной деградации'}")

# 3. ЭФФЕКТ: residual-deep тренируется лучше plain-deep на той же глубине
assert train_acc_gap > 0.03, f"skip не помог обучению (gap={train_acc_gap:+.3f}); центральное утверждение статьи не воспроизвелось"
print(f"[ok] residual-deep > plain-deep по TRAIN acc, gap={train_acc_gap:+.3f}")

# 4. residual-deep возвращает уровень мелкой сети — глубина больше не вредит
recovered = m_res["train_acc"] >= m_shallow["train_acc"] - 0.02
print(f"[{'ok' if recovered else '??'}] residual-deep восстанавливает уровень shallow ({m_res['train_acc']:.3f} vs {m_shallow['train_acc']:.3f})")

## Фигура — degradation и его снятие

Одна картинка на воспроизведённый эффект. Столбики train accuracy: shallow → plain-deep (провал) → residual-deep (восстановление). Сравни с Fig. 4 статьи: та же качественная форма — глубина вредит plain и не вредит residual.

In [ ]:
import matplotlib.pyplot as plt
names = ["shallow\n(1 blk)", f"plain-deep\n({DEEP_BLOCKS} blk)", f"residual-deep\n({DEEP_BLOCKS} blk)"]
tr = [m_shallow["train_acc"], m_plain["train_acc"], m_res["train_acc"]]
te = [m_shallow["test_acc"], m_plain["test_acc"], m_res["test_acc"]]
x = np.arange(3)
plt.figure(figsize=(7, 3.5))
plt.bar(x - 0.2, tr, 0.4, label="train acc")
plt.bar(x + 0.2, te, 0.4, label="test acc", alpha=0.7)
plt.axhline(maj_acc, ls="--", c="gray", label=f"majority {maj_acc:.2f}")
plt.xticks(x, names); plt.ylabel("accuracy"); plt.ylim(0, 1)
plt.title("Degradation и identity-skip · digits · identичный протокол")
plt.legend(); plt.tight_layout(); plt.show()
print(f"gap (train) residual - plain = {m_res['train_acc']-m_plain['train_acc']:+.3f}")

## Пощупать фишку — свип по глубине

Один операционный контраст рискует оказаться удачным. Статья заявляет робастность эффекта по глубине. Прогоним plain vs residual для нескольких значений глубины и посмотрим, как ведёт себя train-gap: у residual accuracy держится, у plain — валится с глубиной.

(Займёт минуту-две — три глубины × два варианта.)

In [ ]:
depths = [6, 12, 18]
plain_tr, res_tr = [], []
for nb in depths:
    set_seed(SEED)
    pm = DeepMLP(IN_DIM, WIDTH, nb, N_CLASSES, residual=False)
    plain_tr.append(train_model(pm, Xtr, ytr, Xte, yte)["train_acc"])
    set_seed(SEED)
    rm = DeepMLP(IN_DIM, WIDTH, nb, N_CLASSES, residual=True)
    res_tr.append(train_model(rm, Xtr, ytr, Xte, yte)["train_acc"])
    print(f"blocks={nb:2d}  plain train_acc={plain_tr[-1]:.3f}  residual train_acc={res_tr[-1]:.3f}  gap={res_tr[-1]-plain_tr[-1]:+.3f}")

plt.figure(figsize=(7, 3.5))
plt.plot(depths, plain_tr, marker="o", label="plain")
plt.plot(depths, res_tr, marker="s", label="residual")
plt.xlabel("n_blocks (глубина)"); plt.ylabel("train accuracy"); plt.ylim(0, 1)
plt.title("Train acc vs глубина: plain деградирует, residual держится")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

❓ **Inline Question 3.** По свипу: у plain train acc падает с глубиной, у residual держится. В какой НОВОЙ ситуации из твоей практики ты потянешься за residual-параметризацией — не обязательно в CNN? (Подумай про любые глубокие/итеративные стеки, где identity — разумный дефолт.)

**Ваш ответ:** …

## Debug-area

Если что-то не сходится, до обучения:
- `DeepMLP(...).eval()` на dummy-батче — форма выхода `[B, 10]`?
- Самый хрупкий узел — `Block.forward`: складываешь ли ты именно вход блока `x`, а не выход первого линейного слоя? Тест на `F(x)=0 -> relu(x)` ловит это.
- Если train-loss residual-сети огромный, но accuracy высокая — это ожидаемо (residual может держать большие логиты); мы меряем именно accuracy, не loss.

## Рефлексия

**Что забрать в работу:** …

**В какой новой ситуации residual — правильный инструмент:** где целевое отображение близко к тождеству/малому возмущению входа, а стек глубокий (итеративные уточнения, deep нормализующие потоки, любые «предскажи дельту, не абсолют»). …

**Что осталось непонятным → следующий узел beam:** почему train-*loss* у residual бывает выше при более высокой accuracy? (связь калибровки логитов и BN в остаточной ветви) …